### Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [253]:
import matplotlib.pyplot as plt
import numpy as np

import os
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Tuple, Optional
import sys

sys.path.append(os.path.abspath('..'))
import time

import jax
import jax.numpy as jnp
jnp.set_printoptions(precision=15, suppress=True)
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_disable_jit", True)
import numpy as np

from floris.floris.floris_model import FlorisModel, TimeSeries

from diffwake.diffwake_jax.turbine.operation_models import power
from diffwake.diffwake_jax.model import load_input, create_state
from diffwake.diffwake_jax import model
from diffwake.diffwake_jax.yaw_runner import make_yaw_runner
from diffwake.diffwake_jax.util import average_velocity_jax
from diffwake.diffwake_jax.turbine.operation_models import power as power_fn
from diffwake.diffwake_jax.simulator import simulate

## Use a 2-turbine debugging layout in JAX to view the calculations

In [254]:
def build_state_runner(
        data_dir: Path,
        farm_yaml: str,
        turbine_yaml: str,
        wind_dir_rad: jax.Array,
        wind_speed: jax.Array,
        turb_intensity: jax.Array,
        dtype,
):
    cfg = load_input(
        str(data_dir / farm_yaml),
        str(data_dir / turbine_yaml),
    ).set(
        wind_directions=wind_dir_rad,
        wind_speeds=wind_speed,
        turbulence_intensities=turb_intensity,
    )
    state = create_state(cfg)
    runner = make_yaw_runner(state)

    x0 = jnp.asarray(cfg.layout["layout_x"], dtype=dtype)
    N = int(x0.shape[0])

    return state, runner, N

DTYPE = jnp.float64
DATA_DIR = Path(rf"data/debug")
FARM_YAML = "gch.yaml"
TURBINE_YAML = "vestas_v802MW.yaml"
WIND_DIR_RAD = jnp.deg2rad(jnp.array([270.0], dtype=DTYPE))
WIND_SPEED = jnp.asarray([9.6], dtype=DTYPE)
TURBULENCE = jnp.full_like(WIND_DIR_RAD, 0.06, dtype=DTYPE)


state, runner, N = build_state_runner(DATA_DIR, FARM_YAML, TURBINE_YAML, WIND_DIR_RAD, WIND_SPEED, TURBULENCE, DTYPE)
yaw_val = jnp.deg2rad(10.0)
yaw_angles = jnp.full((1, N), yaw_val, dtype=DTYPE)
baseline_yaw = jnp.full((1, N), 0.0, dtype=DTYPE)
# print("Yawed run")
# out = runner(yaw_angles)


## FLORIS

In [255]:
fmodel = FlorisModel(DATA_DIR / FARM_YAML)
time_series = TimeSeries(wind_speeds=np.array([9.6]),
                   wind_directions=np.array([270.0]),
                   turbulence_intensities=0.06)
fmodel.set(wind_data=time_series)
fyaw = np.array([[10.0, 10.0]])
fbaseyaw = np.array([[0.0, 0.0]])

print("FLORIS unyawed run")
fmodel.set(yaw_angles=fbaseyaw)
fmodel.run()
fpower_base = fmodel.get_turbine_powers()

print("\nDiffWake unyawed run")
baseline = runner(baseline_yaw)
# print("\nYawed run")
# fmodel.set(yaw_angles=fyaw)
# fmodel.run()
# fpower_yaw = fmodel.get_turbine_powers()
# fpower_yaw_expected = fmodel.get_expected_turbine_powers()

FLORIS unyawed run
val: [[0.000007616639142]], y: [[0.000218200638444]]
val: [[0.027599107518211]], y: [[0.790756599199817]]

DiffWake unyawed run
val: [[0.000007616639142]], y: [[0.000003808319571]]
Yaw eff: [[[[0.000003808319571]]]]
val: [[0.02760324682365]], y: [[0.013803376679437]]
Yaw eff: [[[[0.013803376679437]]]]


In [213]:
# yaw_pow = power(
#     state.farm.power_thrust_table,
#     out.u_sorted,
#     state.flow.air_density,
#     yaw_angles=yaw_angles
# )
baseline_pow = power(
    state.farm.power_thrust_table,
    baseline.u_sorted,
    state.flow.air_density,
    yaw_angles=baseline_yaw
)
# print(f"DiffWake turbine powers (all yawed to {jnp.rad2deg(yaw_val)} degrees): {yaw_pow}")
# print(f"FLORIS turbine powers (all yawed to {fyaw[0,0]} degrees): {fpower_yaw}")
# print(f"Expected FLORIS turbine powers: {fpower_yaw_expected}\n")

#
print(f"DiffWake turbine powers (unyawed): {baseline_pow}")
print(f"FLORIS turbine powers (unyawed): {fpower_base}")


DiffWake turbine powers (unyawed): [[1195849.3993412955   521072.24326432403]]
FLORIS turbine powers (unyawed): [[1195849.3993412955  521071.9140125757]]


### Results
Clearly from the unyawed case, waked turbines are handled differently

In [90]:
# DiffWake final velocity grid
diffwake_vel = baseline.u_sorted
diffwake_avg_vel = average_velocity_jax(diffwake_vel)
diffwake_vel

Array([[[[9.2201050395, 9.6         , 9.8939239853],
         [9.2201050395, 9.6         , 9.8939239853],
         [9.2201050395, 9.6         , 9.8939239853]],

        [[7.2152933481, 7.1526846665, 7.7425976832],
         [6.8703772032, 6.7316383538, 7.372474555 ],
         [7.2165578489, 7.1542282689, 7.7439545956]]]], dtype=float64)

In [91]:
fmodel.core.flow_field.u_sorted

array([[[[9.2201050395, 9.6         , 9.8939239853],
         [9.2201050395, 9.6         , 9.8939239853],
         [9.2201050395, 9.6         , 9.8939239853]],

        [[7.2159135275, 7.1534415539, 7.7432631862],
         [6.8703757001, 6.7316363098, 7.3724729421],
         [7.2159355973, 7.1534684949, 7.7432868689]]]])